In [7]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken
from autogen_agentchat.ui import Console

In [8]:
# 1. LOCAL BRAIN (Ollama Setup)
model_client = OpenAIChatCompletionClient(
    model="llama3.2",
    api_key="placeholder",
    base_url="http://localhost:11434/v1",
    model_info={
        "vision": False,
        "function_calling": True, # CRITICAL: Allows the AI to use tools
        "json_output": True,
        "family": "llama3",
    }
)

In [9]:
from typing import Any # Add this import at the top of your file

# 2. THE TOOL (The Resilient AI Hands)
async def web_search(query: Any) -> str:
    """
    Finds exact information on the web. Always use this for factual queries.
    """
    # 1. Clean the input if the LLM hallucinated a dictionary
    if isinstance(query, dict):
        # Extract the first available value from the dictionary
        clean_query = list(query.values())[0] if query else str(query)
    else:
        clean_query = str(query)
        
    print(f"\n[TOOL TRIGGERED] Searching local database for: {clean_query}...\n")
    return "The Labrador Retriever is a British breed of retriever gun dog. It is known for its friendly demeanor."

# 2. THE TOOL (The AI's Hands)
async def web_search(query: str) -> str:
    """Finds exact information on the web. Always use this for factual queries."""
    print(f"\n[TOOL TRIGGERED] Searching local database for: {query}...\n")
    return "The Labrador Retriever is a British breed of retriever gun dog. It is known for its friendly demeanor."

In [10]:
# 3. THE AGENT 
agent = AssistantAgent(
    name='research_assistant',
    model_client=model_client,
    tools=[web_search],
    system_message='You are a helpful researcher. If you need facts, ALWAYS use the web_search tool.'
)

In [11]:
# 4. STREAMING EXECUTION
async def main():
    print("--- Starting Streaming Interaction ---")
    
    # Task 1: Ask it a factual question requiring the tool
    message_1 = TextMessage(content='Find information about the Labrador Retriever breed.', source='User')
    
    await Console(
        agent.on_messages_stream(
            messages=[message_1],
            cancellation_token=CancellationToken()
        ),
        output_stats=True 
    )

    print("\n--- Testing Agent Memory ---")
    
    # Task 2: Test if it remembers the conversation
    message_2 = TextMessage(content='What was the specific breed I just asked you about?', source='User')
    
    await Console(
        agent.on_messages_stream(
            messages=[message_2],
            cancellation_token=CancellationToken()
        ),
        output_stats=True
    )

# THE JUPYTER FIX: Just await the function directly
await main()

--- Starting Streaming Interaction ---
---------- ToolCallRequestEvent (research_assistant) ----------
[FunctionCall(id='call_r5hjknwp', arguments='{"query":{"description":"Labrador Retriever breed"}}', name='web_search')]
[Prompt tokens: 179, Completion tokens: 26]

[TOOL TRIGGERED] Searching local database for: Labrador Retriever breed...

---------- ToolCallExecutionEvent (research_assistant) ----------
[FunctionExecutionResult(content='The Labrador Retriever is a British breed of retriever gun dog. It is known for its friendly demeanor.', name='web_search', call_id='call_r5hjknwp', is_error=False)]
---------- research_assistant ----------
The Labrador Retriever is a British breed of retriever gun dog. It is known for its friendly demeanor.
---------- Summary ----------
Number of inner messages: 2
Total prompt tokens: 179
Total completion tokens: 26
Duration: 4.99 seconds

--- Testing Agent Memory ---
---------- ToolCallRequestEvent (research_assistant) ----------
[FunctionCall(id='